# 🌿 AgriScan — Plant Disease Detection (TensorFlow / TFLite)

This notebook trains a **MobileNetV2-based CNN** on tomato plant disease images and exports it as a `.tflite` model for on-device inference in the AgriScan Flutter app.

**Classes covered:**
- Tomato___Bacterial_spot
- Tomato___Late_blight
- Tomato___Early_blight
- Tomato___Leaf_Mold
- Tomato___Septoria_leaf_spot
- Tomato___Spider_mites
- Tomato___Target_Spot
- Tomato___Tomato_Yellow_Leaf_Curl_Virus
- Tomato___Tomato_mosaic_virus
- Tomato___healthy

**References:**
- Dataset: [PlantVillage on Kaggle](https://www.kaggle.com/datasets/emmarex/plantdisease)
- Architecture: MobileNetV2, ResNet18, EfficientNetB0

## ⚙️ Step 1: Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install any missing libraries
!pip install -q tensorflow scikit-learn seaborn matplotlib Pillow

import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 📁 Step 2: Dataset Path Configuration

In [ ]:
import os

# ======================================================
# 🔧 CONFIGURE THESE PATHS TO YOUR GOOGLE DRIVE DATASET
# Upload the PlantVillage dataset zip to your Drive and
# update DATASET_PATH to point to the extracted folder.
# ======================================================
DATASET_ZIP  = '/content/drive/MyDrive/agriscan/PlantVillage.zip'  # <-- update path
DATASET_PATH = '/content/PlantVillage/'                              # extraction destination
MODEL_SAVE_PATH  = '/content/drive/MyDrive/agriscan/saved_model/'
TFLITE_SAVE_PATH = '/content/drive/MyDrive/agriscan/model.tflite'
LABELS_SAVE_PATH = '/content/drive/MyDrive/agriscan/labels.txt'

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
print('Paths configured.')

In [ ]:
# Extract dataset from zip if not yet extracted
import zipfile

if not os.path.exists(DATASET_PATH):
    print('Extracting dataset... (this may take a few minutes)')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall('/content/')
    print('Extraction complete!')
else:
    print('Dataset already extracted.')

# List classes found
classes = sorted(os.listdir(DATASET_PATH))
print(f'Found {len(classes)} classes:')
for c in classes:
    n = len(os.listdir(os.path.join(DATASET_PATH, c)))
    print(f'  {c}: {n} images')

## 🔬 Step 3: Data Loading with Augmentation & Class Balancing

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = 224  # MobileNetV2 input size
BATCH_SIZE = 32
SEED       = 42

# Training augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation — no augmentation, only rescale
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Training samples: {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')

In [ ]:
# ---- Class Balancing via class_weight ----
from sklearn.utils.class_weight import compute_class_weight

labels = train_gen.classes
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = dict(enumerate(class_weights_arr))
print('Class weights computed:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  [{i}] {name}: {class_weight_dict[i]:.3f}')

## 🧠 Step 4: Model Architecture — MobileNetV2 Transfer Learning

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers

def build_mobilenetv2_model(num_classes, img_size=224, dropout_rate=0.3):
    base_model = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    # Freeze base model initially
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = build_mobilenetv2_model(NUM_CLASSES)
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 🎯 Step 5: Training — Phase 1 (Frozen Base)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH + 'best_phase1.h5', save_best_only=True, monitor='val_accuracy', verbose=1)
]

EPOCHS_PHASE1 = 15

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    class_weight=class_weight_dict
)

print('Phase 1 training complete!')

## 🔓 Step 6: Fine-Tuning — Phase 2 (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),  # Very low LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH + 'best_phase2.h5', save_best_only=True, monitor='val_accuracy', verbose=1)
]

EPOCHS_PHASE2 = 20

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    class_weight=class_weight_dict
)

print('Phase 2 fine-tuning complete!')

## 📊 Step 7: Full Metrics Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# --- Training History Plots ---
def plot_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(acc, label='Train Accuracy')
    axes[0].plot(val_acc, label='Val Accuracy')
    axes[0].set_title('Model Accuracy'); axes[0].legend()
    axes[1].plot(loss, label='Train Loss')
    axes[1].plot(val_loss, label='Val Loss')
    axes[1].set_title('Model Loss'); axes[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history1, history2)

In [ ]:
# Generate predictions on validation set
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_gen.classes[:len(y_pred)]

# Scalar metrics
print('=== AgriScan Model Evaluation ===')
print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision: {precision_score(y_true, y_pred, average="weighted"):.4f}')
print(f'Recall:    {recall_score(y_true, y_pred, average="weighted"):.4f}')
print(f'F1-Score:  {f1_score(y_true, y_pred, average="weighted"):.4f}')

# Per-class report
print('\n=== Per-Class Classification Report ===')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
short_names = [c.replace('Tomato___', '') for c in CLASS_NAMES]
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=short_names, yticklabels=short_names, ax=ax
)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — AgriScan MobileNetV2')
plt.tight_layout(); plt.show()

## 📱 Step 8: Export to TFLite for Flutter On-Device Inference

In [ ]:
import time

# ---- Save Keras Model ----
model.save(MODEL_SAVE_PATH + 'agriscan_model')
print('Keras SavedModel saved.')

# ---- Convert to TFLite ----
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optional: Quantization (reduces size, speeds up mobile inference)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save the .tflite file
with open(TFLITE_SAVE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_SAVE_PATH) / 1024
print(f'\n✅ TFLite model saved: {TFLITE_SAVE_PATH}')
print(f'   Model size: {size_kb:.1f} KB')

In [ ]:
# ---- Save Labels File ----
with open(LABELS_SAVE_PATH, 'w') as f:
    for name in CLASS_NAMES:
        f.write(name + '\n')
print(f'✅ Labels saved: {LABELS_SAVE_PATH}')
print('\nClass labels:')
for i, n in enumerate(CLASS_NAMES):
    print(f'  {i}: {n}')

## 🔍 Step 9: TFLite Inference Speed Test

In [ ]:
# Benchmark TFLite inference speed
interpreter = tf.lite.Interpreter(model_path=TFLITE_SAVE_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Dummy image
dummy_input = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy_input)

# Warm-up run
interpreter.invoke()

# Timed run (average of 20)
times = []
for _ in range(20):
    t0 = time.perf_counter()
    interpreter.invoke()
    times.append(time.perf_counter() - t0)

avg_ms = np.mean(times) * 1000
print(f'TFLite average inference time: {avg_ms:.2f} ms')
output = interpreter.get_tensor(output_details[0]['index'])
pred_idx = np.argmax(output)
print(f'Sample prediction: {CLASS_NAMES[pred_idx]} ({output[0][pred_idx]*100:.1f}%)')

---
## ✅ Summary

After running all cells:
1. Download `model.tflite` and `labels.txt` from your Google Drive.
2. Place them in the Flutter app at:
   - `app/assets/ml/model.tflite`
   - `app/assets/ml/labels.txt`
3. The Flutter app will use `tflite_flutter` to load and run inference on camera images.